#Data Loading, Cleaning & Setup

In [1]:
!pip install datasets

In [2]:
from datasets import load_dataset

In [3]:
dataset =load_dataset("CShorten/ML-ArXiv-Papers",split='train')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
import pandas as pd

In [5]:
df=pd.DataFrame(dataset)
df

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...,...,...
117587,4995,NaN,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,4996,NaN,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,4997,NaN,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,4998,NaN,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [6]:
df=df[['title','abstract']]

In [7]:
df

,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...
117587,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [8]:
df.shape

(117592, 2)

In [9]:
df = df.head(15000)
df.shape

(15000, 2)

In [10]:
df.isnull().sum()

,0
title,0
abstract,0


In [11]:
df["paper_text"] = df["title"]+" "+df["abstract"]

/tmp/ipykernel_19199/1043561184.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"] = df["title"]+" "+df["abstract"]


In [12]:
df[["paper_text"]].head()

,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [13]:
from sentence_transformers import SentenceTransformer

In [14]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [15]:
df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
df["paper_text"]=df["paper_text"].str.strip()

/tmp/ipykernel_19199/2190359946.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
/tmp/ipykernel_19199/2190359946.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["paper_text"].str.strip()


In [16]:
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
sample_embedding = model.encode(
    df["paper_text"].head(5).tolist()
)

print(type(sample_embedding))
print(sample_embedding.shape)

<class 'numpy.ndarray'>
(5, 384)


In [18]:
similarity = cosine_similarity(sample_embedding[0].reshape(1,-1),
                               sample_embedding[0].reshape(1,-1))
print(similarity)

[[1.0000001]]


In [19]:
similarity = cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[1].reshape(1,-1))
print(similarity)

[[0.36625272]]


In [20]:
for i in range(1,5):
  sim = cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[i].reshape(1,-1))
  print(sim)

[[0.36625272]]
[[0.33522844]]
[[0.15505108]]
[[0.37421533]]


#Generating Full Embeddings

In [21]:
import os
import numpy as np
if os.path.exists("paper_embeddings.npy"):
    print("Loading saved embeddings")
    embeddings = np.load("paper_embeddings.npy")
else:
    print("Generating embeddings")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    np.save("paper_embeddings.npy", embeddings)
    print("Embeddings saved successfully!")

Loading saved embeddings


In [22]:
!pip install faiss-cpu


In [23]:
import faiss

In [24]:
if os.path.exists("paper_faiss.index"):
    print("Loading existing FAISS index")
    index = faiss.read_index("paper_faiss.index")
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(384)
    index.add(embeddings)
    faiss.write_index(index, "paper_faiss.index")
    print("FAISS index saved successfully!")

Loading existing FAISS index


In [25]:
query = "deep learning for medical image analysis"
query_embedding = model.encode([query])
query_embedding.shape

(1, 384)

In [26]:
faiss.normalize_L2(query_embedding)

In [27]:
D,I = index.search(query_embedding,5)
print(D)
print(I)

[[0.6807244  0.67092204 0.65219975 0.62811744 0.61311525]]
[[10466 13730 11873 12691 11282]]


In [28]:
print(df.iloc[10466]["title"])

A Perspective on Deep Imaging


In [29]:
print(df.iloc[10466]["abstract"])

  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance in clinical and
preclinical applications. To realize the full impact of machine learning on
medical imaging, major challenges must be addressed.



In [30]:
print(df.iloc[11873]["title"])

Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection


In [31]:
def search_paper(query,k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I = index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity Score :",score)
    print("Title :",df.iloc[idx]["title"])
    print("Abstract :",df.iloc[idx]["abstract"][:500])
    print()

In [32]:
search_paper("deep learning for medical image analysis")

Similarity Score : 0.6807244
Title : A Perspective on Deep Imaging
Abstract :   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Similarity Score : 0.67092204
Title : Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Abstract :   Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-trained using, for

In [33]:
!pip install transformers==4.46.3

In [34]:
from transformers import pipeline
summarizer = pipeline(
    "summarization",
    model = "facebook/bart-large-cnn",
)

In [35]:
summary=summarizer(df.iloc[10466]["abstract"],max_length = 120 , min_length = 40)
print(summary[0]["summary_text"])

The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.


In [36]:
summary[0]["summary_text"]

'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'

In [37]:
for score,idx in zip(D[0],I[0]):
    print("Similarity Score :",score)
    print("Title :",df.iloc[idx]["title"])
    print("Abstract :",df.iloc[idx]["abstract"][:500])
    summary=summarizer(df.iloc[idx]["abstract"],max_length = 120 , min_length = 40)
    print(summary[0]["summary_text"])
    print()

Similarity Score : 0.6807244
Title : A Perspective on Deep Imaging
Abstract :   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance
The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Similarity Score : 0.67092204
Title : Convolutional Neural Networks for Medical

In [38]:
!pip install keybert==0.8.5

In [39]:
from keybert import KeyBERT

In [40]:
kw_model = KeyBERT(model)


In [41]:
def search_and_summarize(query, k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D, I = index.search(query_embedding, k)
  for score, idx in zip(D[0], I[0]):
    print("Similarity score :", score)
    print("Title :", df.iloc[idx]["title"])
    print("Abstract :", df.iloc[idx]["abstract"][:500])
    print()
    summary = summarizer(df.iloc[idx]["abstract"], max_length=120, min_length=40, do_sample=False)
    print(summary[0]["summary_text"])
    print()
    abstract = df.iloc[idx]["abstract"]
    keywords = kw_model.extract_keywords(abstract, keyphrase_ngram_range=(1,3), stop_words="english")
    print("Keywords:")
    for k, s in keywords:
      if s > 0.5:
        print(" •", k)
    print()

In [42]:
search_and_summarize("Deep learning in medical imaging",k=5)

Similarity score : 0.73549855
Title : A Perspective on Deep Imaging
Abstract :   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Keywords:
 • tomographic imaging deep
 • imaging deep learning
 • learning me

#Named Entity Recognition

In [43]:
import re

In [44]:
frameworks = [
    # Deep Learning Frameworks
    'tensorflow', 'keras', 'pytorch', 'jax', 'mxnet', 'caffe',
    'theano', 'paddlepaddle', 'fastai', 'chainer',

    # ML Libraries
    'scikit-learn', 'xgboost', 'lightgbm', 'catboost', 'statsmodels',

    # NLP Libraries
    'huggingface', 'spacy', 'nltk', 'gensim', 'transformers',

    # Computer Vision
    'opencv', 'pillow', 'torchvision', 'albumentations',

    # Data & Compute
    'numpy', 'pandas', 'scipy', 'cupy', 'dask', 'spark',

    # Search & Retrieval
    'faiss', 'elasticsearch', 'annoy',

    # Web & API
    'flask', 'fastapi', 'streamlit', 'django',

    # Other
    'tensorboard', 'wandb', 'mlflow', 'ray', 'celery'
]

In [45]:
models = ['bert', 'gpt', 'transformer', 'resnet', 'yolo', 'lstm', 'cnn', 'alexnet', 'roberta', 'llama', 'vgg', 'gan', 'rnn', 'efficientnet', 'clip', 'dalle', 'mistral', 'gemini', 'vit', 'unet', 't5', 'gpt2', 'gpt3', 'gpt4', 'wavenet', 'word2vec', 'glove']

In [46]:
programming_languages = ['python', 'r', 'java', 'c++', 'javascript', 'go', 'julia', 'matlab', 'scala', 'rust', 'cuda', 'sql']

In [47]:
organizations = ['google', 'microsoft', 'ibm', 'facebook', 'amazon', 'apple', 'openai', 'meta', 'nvidia', 'deepmind', 'anthropic', 'hugging face', 'stanford', 'mit', 'berkeley', 'oxford', 'cambridge']

In [48]:
entity_descriptions = {
    # Frameworks
    'tensorflow': 'ML framework by Google',
    'pytorch': 'ML framework by Facebook',
    'keras': 'High level neural network API',
    'scikit-learn': 'ML library for Python',
    'huggingface': 'NLP models and datasets platform',
    'opencv': 'Computer vision library',
    'xgboost': 'Gradient boosting framework',
    'faiss': 'Similarity search library by Facebook',
    'spacy': 'NLP library for Python',
    'nltk': 'Natural language toolkit',

    # Models
    'bert': 'Language model by Google for NLP',
    'gpt': 'Language model by OpenAI',
    'resnet': 'Image classification model',
    'cnn': 'Convolutional Neural Network',
    'lstm': 'Long Short Term Memory network',
    'transformer': 'Attention based neural network',
    'yolo': 'Real time object detection model',
    'gan': 'Generative Adversarial Network',
    'vgg': 'Deep CNN for image recognition',
    'unet': 'Image segmentation model',

    # Languages
    'python': 'Popular programming language for ML',
    'r': 'Statistical programming language',
    'java': 'Object oriented programming language',
    'matlab': 'Numerical computing environment',
    'cuda': 'Parallel computing platform by Nvidia',

    # Organizations
    'google': 'AI research and technology company',
    'microsoft': 'Technology and AI company',
    'facebook': 'Social media and AI research company',
    'openai': 'AI safety and research company',
    'nvidia': 'GPU and AI hardware company',
    'deepmind': 'AI research lab by Google',
    'meta': 'AI research and social media company',
    'stanford': 'Leading AI research university',
    'mit': 'Leading technology university',
}

In [49]:
def classify_query_entities(query):
    query_lower = query.lower()
    found_entities = {}
    for fw in frameworks:
        if re.search(r'\b' + re.escape(fw) + r'\b', query_lower):
            desc = entity_descriptions.get(fw, '')
            found_entities[fw] = ('Framework', desc)
    for m in models:
        if re.search(r'\b' + re.escape(m) + r'\b', query_lower):
            desc = entity_descriptions.get(m, '')
            found_entities[m] = ('Model', desc)
    for lang in programming_languages:
        if re.search(r'\b' + re.escape(lang) + r'\b', query_lower):
            desc = entity_descriptions.get(lang, '')
            found_entities[lang] = ('Programming Language', desc)
    for org in organizations:
        if re.search(r'\b' + re.escape(org) + r'\b', query_lower):
            desc = entity_descriptions.get(org, '')
            found_entities[org] = ('Organization', desc)
    return found_entities

In [50]:
entities = classify_query_entities("research using pytorch and BERT from google")
print(entities)

{'pytorch': ('Framework', 'ML framework by Facebook'), 'bert': ('Model', 'Language model by Google for NLP'), 'google': ('Organization', 'AI research and technology company')}


In [51]:
for entity, entity_type in entities.items():
    print(f"- {entity} : {entity_type}")

- pytorch : ('Framework', 'ML framework by Facebook')
- bert : ('Model', 'Language model by Google for NLP')
- google : ('Organization', 'AI research and technology company')


In [52]:
entities = classify_query_entities("Python on medical dataset with fastapi")
for entity, entity_type in entities.items():
    print(f"- {entity} : {entity_type}")

- fastapi : ('Framework', '')
- python : ('Programming Language', 'Popular programming language for ML')


In [53]:
def search_and_summarize_with_ner(query, k=5):

    # NER on query
    print("Query :", query)
    print()
    entities = classify_query_entities(query)
    if entities:
        print("Detected Entities:")
        for entity, (entity_type, desc) in entities.items():
            if desc:
                print(f"  - {entity} : {entity_type} → {desc}")
            else:
                print(f"  - {entity} : {entity_type}")
    else:
        print("No technical entities detected")
    print()

    # Search + Summarize + Keywords
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    for score, idx in zip(D[0], I[0]):
        print("Similarity score :", score)
        print("Title :", df.iloc[idx]["title"])
        print("Abstract :", df.iloc[idx]["abstract"][:500])
        print()
        summary = summarizer(df.iloc[idx]["abstract"], max_length=120, min_length=40, do_sample=False)
        print(summary[0]["summary_text"])
        print()
        abstract = df.iloc[idx]["abstract"]
        keywords = kw_model.extract_keywords(abstract, keyphrase_ngram_range=(1,3), stop_words="english")
        print("Keywords:")
        for k, s in keywords:
            if s > 0.5:
                entities_in_keyword = classify_query_entities(k)
                if entities_in_keyword:
                    tags = ", ".join([f"{e}: {t}" for e, (t, d) in entities_in_keyword.items()])
                    print(f" • {k}  [{tags}]")
                else:
                    print(f" • {k}")
        print("-" * 60)

In [54]:
search_and_summarize_with_ner("research using pytorch and BERT from google", k=3)

Query : research using pytorch and BERT from google

Detected Entities:
  - pytorch : Framework → ML framework by Facebook
  - bert : Model → Language model by Google for NLP
  - google : Organization → AI research and technology company

Similarity score : 0.4507684
Title : A Tour of TensorFlow
Abstract :   Deep learning is a branch of artificial intelligence employing deep neural
network architectures that has significantly advanced the state-of-the-art in
computer vision, speech recognition, natural language processing and other
domains. In November 2015, Google released $\textit{TensorFlow}$, an open
source deep learning software library for defining, training and deploying
machine learning models. In this paper, we review TensorFlow and put it in
context of modern deep learning concepts and s

Deep learning is a branch of artificial intelligence employing deep neuralnetwork architectures. In November 2015, Google released TensorFlow, an open-source deep learning software library. 

#Gen AI , LLMs , Agents

In [55]:
!pip install transformers -q

In [56]:
from transformers import pipeline

In [57]:
hf_ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [58]:
result = search_paper("TensorFlow deep learning software")

Similarity Score : 0.7714709
Title : A Tour of TensorFlow
Abstract :   Deep learning is a branch of artificial intelligence employing deep neural
network architectures that has significantly advanced the state-of-the-art in
computer vision, speech recognition, natural language processing and other
domains. In November 2015, Google released $\textit{TensorFlow}$, an open
source deep learning software library for defining, training and deploying
machine learning models. In this paper, we review TensorFlow and put it in
context of modern deep learning concepts and s

Similarity Score : 0.7505419
Title : TensorLayer: A Versatile Library for Efficient Deep Learning Development
Abstract :   Deep learning has enabled major advances in the fields of computer vision,
natural language processing, and multimedia among many others. Developing a
deep learning system is arduous and complex, as it involves constructing neural
network architectures, managing training/trained models, tuning optimizatio

In [59]:
idx = df[df["title"]=="A Tour of TensorFlow"].index[0]

In [60]:
print(idx)

10675


In [61]:
text = df.iloc[10675]["abstract"]

In [62]:
clean_text = re.sub(r'\$\\textit\{|\}\$|\\emph\{|\}', '', text)

In [63]:
print(clean_text[:500])

  Deep learning is a branch of artificial intelligence employing deep neural
network architectures that has significantly advanced the state-of-the-art in
computer vision, speech recognition, natural language processing and other
domains. In November 2015, Google released TensorFlow, an open
source deep learning software library for defining, training and deploying
machine learning models. In this paper, we review TensorFlow and put it in
context of modern deep learning concepts and software. We


In [64]:
entities = hf_ner(clean_text)
print(entities)

[{'entity_group': 'ORG', 'score': np.float32(0.99808466), 'word': 'Google', 'start': 257, 'end': 263}, {'entity_group': 'MISC', 'score': np.float32(0.8605759), 'word': 'TensorFlow', 'start': 273, 'end': 283}, {'entity_group': 'ORG', 'score': np.float32(0.6932915), 'word': 'Ten', 'start': 418, 'end': 421}, {'entity_group': 'MISC', 'score': np.float32(0.52699655), 'word': '##sor', 'start': 421, 'end': 424}, {'entity_group': 'ORG', 'score': np.float32(0.62468284), 'word': '##F', 'start': 424, 'end': 425}, {'entity_group': 'MISC', 'score': np.float32(0.7692129), 'word': '##low', 'start': 425, 'end': 428}, {'entity_group': 'MISC', 'score': np.float32(0.52440953), 'word': 'Tensor', 'start': 666, 'end': 672}, {'entity_group': 'ORG', 'score': np.float32(0.59629315), 'word': '##F', 'start': 672, 'end': 673}, {'entity_group': 'MISC', 'score': np.float32(0.5305284), 'word': '##low', 'start': 673, 'end': 676}, {'entity_group': 'ORG', 'score': np.float32(0.6419362), 'word': 'Theano', 'start': 710, 

In [65]:
def merge_entities(text, title=""):
    result = classify_query_entities(text)
    hf_result = hf_ner(text)
    skip_words = ["network", "perceptron", "convolutional", "recurrent"]
    known_datasets = ["daquar", "timit", "mnist", "imagenet", "cifar", "vqa"]
    for ent in hf_result:
        word = ent['word']
        if word.startswith("##"):
            continue
        if len(word) < 4:
            continue
        if ent['score'] < 0.95:
            continue
        if word.lower() in title.lower():
            continue
        if any(sw in word.lower() for sw in skip_words):
            continue
        if word.lower() in known_datasets:
            continue
        if ent['entity_group'] == 'ORG':
            word_lower = word.lower()
            if word_lower not in result:
                result[word_lower] = ('Organization', 'Detected by HuggingFace NER')
    return result

In [66]:
def search_and_summarize_hybrid(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    for score, idx in zip(D[0], I[0]):
        abstract = df.iloc[idx]["abstract"]
        title = df.iloc[idx]["title"]
        clean_abstract = re.sub(r'\$\\textit\{|\}\$|\\emph\{|\}', '', abstract)
        print("Similarity score :", score)
        print("Title :", title)
        print("Abstract :", abstract[:500])
        print()
        summary = summarizer(abstract, max_length=120, min_length=40, do_sample=False)
        print(summary[0]["summary_text"])
        print()
        entities = merge_entities(clean_abstract, title)
        print("Entities:")
        for entity, (etype, desc) in entities.items():
            print(" •", entity, ":", etype)
        print("-" * 60)

In [67]:
search_and_summarize_hybrid("deep learning frameworks for computer vision", k=3)

Similarity score : 0.59719425
Title : LightNet: A Versatile, Standalone Matlab-based Environment for Deep
  Learning
Abstract :   LightNet is a lightweight, versatile and purely Matlab-based deep learning
framework. The idea underlying its design is to provide an easy-to-understand,
easy-to-use and efficient computational platform for deep learning research.
The implemented framework supports major deep learning architectures such as
Multilayer Perceptron Networks (MLP), Convolutional Neural Networks (CNN) and
Recurrent Neural Networks (RNN). The framework also supports both CPU and GPU
computation, and the switch betwee

The LightNet is a lightweight, versatile and purely Matlab-based deep learning framework. It is designed to be easy-to-use and efficient. It can be used for computer vision, natural language processing and other applications.

Entities:
 • cnn : Model
 • rnn : Model
 • matlab : Programming Language
------------------------------------------------------------
Similarit

In [68]:
from google.colab import userdata
key = userdata.get("GROQ_API_KEY")
print(key[:8])

gsk_eGRW


In [69]:
!pip install langchain langchain-groq -q

In [70]:
from langchain_groq import ChatGroq

In [71]:
llm = ChatGroq(groq_api_key=key, model="llama-3.1-8b-instant")

In [72]:
response = llm.invoke("Say hello in one sentence")

In [73]:
print(response.content)

Hello, how are you today?


In [74]:
!pip install langchain-community -q

In [75]:
from langchain.tools import tool

In [76]:
@tool
def search_papers_tool(query: str) -> str:
    """Search for research papers on a topic and return summaries with entities."""
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, 3)
    output = ""
    for score, idx in zip(D[0], I[0]):
        abstract = df.iloc[idx]["abstract"]
        title = df.iloc[idx]["title"]
        clean_abstract = re.sub(r'\$\\textit\{|\}\$|\\emph\{|\}', '', abstract)
        summary = summarizer(abstract, max_length=120, min_length=40, do_sample=False)
        entities = merge_entities(clean_abstract, title)
        output += "Title: " + title + "\n"
        output += "Summary: " + summary[0]["summary_text"] + "\n"
        output += "Entities: " + str(entities) + "\n\n"
    return output

In [77]:
@tool
def extract_keywords_tool(text: str) -> str:
    """Extract top keywords from a piece of text like an abstract."""
    keywords = kw_model.extract_keywords(text, keyphrase_ngram_range=(1,3), stop_words="english")
    output = ""
    for word, score in keywords:
        if score > 0.5:
            output += word + "\n"
    return output

In [78]:
@tool
def compare_papers_tool(query: str) -> str:
    """Find two papers on a topic and compare their approaches using the LLM."""
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, 2)

    paper1 = df.iloc[I[0][0]]
    paper2 = df.iloc[I[0][1]]

    comparison_prompt = f"""Compare the following two research papers.

Paper 1
Title: {paper1['title']}
Abstract: {paper1['abstract']}

Paper 2
Title: {paper2['title']}
Abstract: {paper2['abstract']}

Compare them based on:
1. Research Objective
2. Methodology
3. Key Contributions
4. Limitations

Present the comparison in a clear table."""

    response = llm.invoke(comparison_prompt)
    return response.content

In [79]:
!pip install -U langchain langchain-groq -q

In [80]:
import langchain
print(langchain.__version__)

1.3.14


In [81]:
@tool
def explain_paper_tool(title: str) -> str:
    """Explain a research paper in simple, beginner-friendly terms given its title."""
    query_embedding = model.encode([title])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, 1)
    idx = I[0][0]
    paper_title = df.iloc[idx]["title"]
    abstract = df.iloc[idx]["abstract"]

    explain_prompt = f"""Explain this research paper in simple terms, as if explaining to someone new to machine learning.

Title: {paper_title}
Abstract: {abstract}

Explain:
1. What problem does this paper solve?
2. How does it solve it, in simple words?
3. Why does it matter?"""

    response = llm.invoke(explain_prompt)
    return response.content

In [82]:
tools = [search_papers_tool, extract_keywords_tool, compare_papers_tool, explain_paper_tool]

In [83]:
llm_with_tools = llm.bind_tools(tools)

In [84]:
import json
from datetime import datetime

def ask_agent(question):
    try:
        response = llm_with_tools.invoke(question)
        tool_call = response.tool_calls[0]
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        if tool_name == "search_papers_tool":
            result = search_papers_tool.invoke(tool_args["query"])
        elif tool_name == "extract_keywords_tool":
            result = extract_keywords_tool.invoke(tool_args["text"])
        elif tool_name == "compare_papers_tool":
            result = compare_papers_tool.invoke(tool_args["query"])
        elif tool_name == "explain_paper_tool":
            result = explain_paper_tool.invoke(tool_args["title"])

        final_response = llm.invoke(f"Based on this research paper information, answer the user's question: '{question}'\n\n{result}")
        print(final_response.content)
        print()
        print("Tool used:", tool_name)

        log_entry = {
            "timestamp": str(datetime.now()),
            "question": question,
            "tool_used": tool_name,
            "answer": final_response.content
        }
        with open("agent_log.json", "a") as f:
            f.write(json.dumps(log_entry) + "\n")

    except Exception as e:
        print("Sorry, I couldn't process that question. Try rephrasing it.")
        print("Error details:", e)

In [85]:
ask_agent("What are some papers about reinforcement learning?")

Based on the provided research paper information, here are some papers about reinforcement learning:

1. **Selecting the State-Representation in Reinforcement Learning**: This paper considers the problem of selecting the right state-representation in a reinforcement learning problem and proposes several models of the observations.
2. **Reinforcement Learning algorithms for regret minimization in structured Markov Decision Processes**: This paper focuses on the goal of minimizing regret in a finite time horizon in the Reinforcement Learning (RL) framework, particularly in structured Markov Decision Processes (MDPs).
3. **PAC Reinforcement Learning with Rich Observations**: This paper proposes a new model for reinforcement learning with rich observations and provides theoretical justification for reinforcement learning with function approximation.

These papers cover various aspects of reinforcement learning, including state-representation selection, regret minimization, and learning wit

In [86]:
ask_agent("Compare papers about GANs")

Sorry, I couldn't process that question. Try rephrasing it.
Error details: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=compare_papers_tool>{"query": "GANs"}'}}


In [87]:
ask_agent("Extract keywords from this text: Deep reinforcement learning combines neural networks with reinforcement learning principles to solve complex decision-making tasks.")

Based on the given research paper information, the extracted keywords are:

1. Deep reinforcement learning
2. Deep reinforcement
3. Reinforcement learning combines
4. Neural networks reinforcement
5. Reinforcement learning principles

Tool used: extract_keywords_tool


In [88]:
ask_agent("Explain the paper A Tour of TensorFlow in simple terms")

Based on the provided research paper information, I'll explain the paper "A Tour of TensorFlow" in simple terms.

**What does the paper do?**

The paper, "A Tour of TensorFlow," aims to make machine learning easier to use, especially for people who are not experts in the field. It introduces a new tool called TF.Learn, which provides a simple and user-friendly interface to build, train, and test machine learning models.

**How does it work?**

Imagine you're trying to build a machine learning model but don't know where to start. This paper provides a pre-made kit called TF.Learn, which includes all the necessary tools and instructions. TF.Learn is built on top of a powerful tool called TensorFlow, which is used by experts to build complex models. TF.Learn simplifies TensorFlow and presents a simpler interface, making it easier for non-experts to use.

**Why is it important?**

Machine learning is becoming increasingly important in many areas, such as healthcare, finance, and technology

In [89]:
ask_agent("What are some papers about neural networks?")

Your max_length is set to 120, but your input_length is only 59. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=29)
Your max_length is set to 120, but your input_length is only 74. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=37)
Your max_length is set to 120, but your input_length is only 99. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=49)


Based on the provided research paper information, here are some papers related to neural networks:

1. "Neural Networks for Beginners. A fast implementation in Matlab, Torch, TensorFlow" - This paper provides an introduction to neural networks and their implementation in different frameworks.
2. "Neural Networks for Complex Data" - This paper summarizes advances in neural networks for complex data over the last decade.
3. "Deep learning: Technical introduction" - This paper provides a technical introduction to three common forms of neural network architectures and their building blocks.

These papers cover various aspects of neural networks, including their implementation, applications, and technical aspects.

Tool used: search_papers_tool


In [90]:
ask_agent("What papers exist about computer vision?")

Based on the provided research paper information, here are some papers and topics related to computer vision:

1. **"Evolution of active categorical image classification via saccadic eye movement"**: This paper discusses a new technique called "active categorical classifier" (ACC) that selectively scans only a portion of an image, inspired by the saccadic movements of the eye.

2. **"What you need to know about the state-of-the-art computational models of object-vision: A tour through the models"**: This paper provides an overview of the current state-of-the-art computational models of object vision, including their hierarchical structures and training methods using millions of images.

3. **"TasselNet: Counting maize tassels in the wild via local counts regression network"**: This paper focuses on the task of automating the counting of maize tassels in plant phenotyping, using a local counts regression network to achieve accurate and efficient results.

While these papers are not spec

In [91]:
ask_agent("Extract keywords from: attention mechanisms in transformers")

Based on the given research paper information, the extracted keywords related to "attention mechanisms in transformers" are:

1. **Attention Mechanisms**
2. **Transformers**
3. **Attention**
4. **Mechanisms**

These keywords are crucial components of the research paper and are directly related to the topic of "attention mechanisms in transformers."

Tool used: extract_keywords_tool


In [92]:
with open("agent_log.json") as f:
    print(f.read())

{"timestamp": "2026-07-16 19:43:55.269131", "question": "What are some papers about reinforcement learning?", "tool_used": "search_papers_tool", "answer": "Based on the research paper information provided, here are some papers about reinforcement learning:\n\n1. \"Selecting the State-Representation in Reinforcement Learning\" \n   - This paper discusses the problem of selecting the right state-representation in a reinforcement learning problem.\n\n2. \"Reinforcement Learning algorithms for regret minimization in structured Markov Decision Processes\" \n   - This paper focuses on the goal of choosing a policy to maximize the reward collected or minimize the regret incurred in a finite time horizon within the reinforcement learning framework.\n\n3. \"PAC Reinforcement Learning with Rich Observations\" \n   - This paper proposes a new model for reinforcement learning with rich Observations and provides theoretical justification for reinforcing learning with function approximation.\n\nThes